In [11]:
!nvidia-smi

Mon Mar 30 14:04:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti   WDDM  |   00000000:26:00.0  On |                  N/A |
|  0%   39C    P8             27W /  300W |     954MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
KOHYA_BRANCH = "sd3" #@param["main","dev","main_diffusers","sd3"]
LYCORIS_BRANCH = "main" #@param["main","dev"]

# @title ## 1.1. KOHYA Install Dependencies
import os
import zipfile
import shutil
import time

# root_dir
root_dir = "H:/kohya_ssPR"
train_dir= "H:/kohya_ssPR"

hf_token = os.environ["HF_TOKEN"]
user_header = f'"Authorization: Bearer {hf_token}"'


def main():
    os.chdir(train_dir)
    os.chdir(train_dir)
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    os.environ["BITSANDBYTES_NOWELCOME"] = "1"
    os.environ["SAFETENSORS_FAST_GPU"] = "1"

    cuda_path = "/usr/local/cuda-12.2/targets/x86_64-linux/lib"
    ld_library_path = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = f"{ld_library_path}:{cuda_path}"

main()


In [13]:
#nai = r"Naska223/Mystorage/illustrij_v18.safetensors"
nai = r"H:\ComfyUI_windows_portable\ComfyUI\models\Stable-diffusion\ILL\illustrij_v18.safetensors"
dataset_root = r"H:\TRAINING\트레이닝_이미지\모음\류진\33\img\caption"
reg_dataset_root = ""
print(nai)

H:\ComfyUI_windows_portable\ComfyUI\models\Stable-diffusion\ILL\illustrij_v18.safetensors


In [ ]:
#@title CREATE DATASET
import toml
import ast
import shutil
import os


#@markdown * caption drop시 class_token을 대신 사용함
class_token = "" #@param {type:'string'}

data_set_part = "BOTH" #@param["BOTH","FACE","BODY"]

#@markdown * caption_drop = 전체 캡션을 class_token으로 교체 / tag_drop_rate = 캡션중 일부 캡션만
caption_drop_rate = 0.0 #@param{type:'number'}
caption_tag_drop_rate = 0 #@param{type:'number'}
caption_drop_every_epoch = 0 #@param{type:'number'}

random_crop = False
shuffle = False #@param{type:'boolean'}
keep_token = 1 #@param{type:'number'}

#@markdown *  TE: CLIPSdpaAttention,CLIPAttention / CLIPMLP
#@markdown *  UNET: Transformer2DModel(attn-mlp) /CrossAttention,SelfAttention,Attention(attn) / FeedForward (ff)

Lycoris_preset = "attn-mlp" #@param['full','attn-mlp','custom'] {allow-input: true}
Text_Module = "CLIPSdpaAttention,CLIPAttention,CLIPMLP" #@param{type:'string'} #CLIPSdpaAttention,CLIPAttention,CLIPMLP
Text_Name = "none" #@param['.*','attn','text_model.encoder.layers.0,^text_model.encoder.layers.1(?!\d)(..*)?$','']{allow-input: true}
Unet_Module = "Transformer2DModel,ResnetBlock2D,Downsample2D,Upsample2D" #@param['Transformer2DModel','Transformer2DModel,ResnetBlock2D,Downsample2D,Upsample2D']{allow-input: true}
Unet_Name = "none" #@param['.*','attentions','attn','(?!middle_block\b)\b\w+','^(?!.*(ff|proj)).*output_blocks.0.1..*$,^(?!.*(ff|proj)).*input_blocks.8.1..*$,^(?!.*(ff|proj)).*output_blocks.1.1..*$','mid_block.attentions , up_blocks.0.attentions , down_blocks.2.attentions'] middle_block,input_blocks.8,output_blocks.0,output_blocks.1,output_blocks.2

#@markdown * Block target maker (it only works if the preset is custom and Unet_Name is empty)
#@markdown * You can make it with IN4,IN5,IN7,IN8,MID,OUT0,OUT1,OUT2,OUT3,OUT4,OUT5 (SDXL)
Target_Blocks = "MID,OUT0,OUT1,OUT2" #@param{type:'string'}
Target_Elements=".*proj.in, .*proj.out , .*attn , .*ff.net" #@param{type:'string'}

#@markdown * Algo map maker (Lycoris preset should be custom mode to work)
Module_Maps = "" #@param{type:'string'}
Name_Maps = "" #@param{type:'string'}
Mapping_KV= "factor=4" #@param{type:'string'}

#@markdown * resolutions
#1024x1024,896x1152,832x1216,768x1344,640x1536
#768,832,896,960,1024
#768,800,832,864,896,928,960,992,1024
override_res = "1024x1024" # @param {type:"string"}
use_reg=True #@param{type:'boolean'}
#@markdown * batches
batch_size = "2" #@param{type:"string"}
repeat = 10 #@param{type:"number"}
reg_repeat = 1 #@param{type:"number"}

resolutions = override_res.split(',')

target_resolutions=[]
for res in resolutions:
  target_resolutions.append(res)
image_part_path = []
if os.path.exists(dataset_root) and (data_set_part=="BOTH" or data_set_part=="BODY"):
  count = len([f for f in os.listdir(dataset_root) if os.path.isfile(os.path.join(dataset_root, f))])
  if count > 0 :
    image_part_path.append(dataset_root)
if os.path.exists(os.path.join(dataset_root,"body")) and (data_set_part=="BOTH" or data_set_part=="BODY"):
  image_part_path.append(os.path.join(dataset_root,"body"))
if os.path.exists(os.path.join(dataset_root,"justfit")) and (data_set_part=="BOTH" or data_set_part=="BODY"):
  image_part_path.append(os.path.join(dataset_root,"justfit"))
if os.path.exists(os.path.join(dataset_root,"horizon")) and (data_set_part=="BOTH" or data_set_part=="BODY"):
  image_part_path.append(os.path.join(dataset_root,"horizon"))
if os.path.exists(os.path.join(dataset_root,"vertical")) and (data_set_part=="BOTH" or data_set_part=="FACE"):
  image_part_path.append(os.path.join(dataset_root,"vertical"))
if os.path.exists(os.path.join(dataset_root,"square")) and (data_set_part=="BOTH" or data_set_part=="FACE"):
  image_part_path.append(os.path.join(dataset_root,"square"))

#make toml dataset
toml_raw = {}
toml_raw['general']={}
toml_raw['general']['caption_extension'] = ".txt"
toml_raw['general']['caption_dropout_rate'] = caption_drop_rate
toml_raw['general']['caption_tag_dropout_rate'] = caption_tag_drop_rate
toml_raw['general']['caption_dropout_every_n_epochs'] = caption_drop_every_epoch
toml_raw['general']['random_crop'] = random_crop
toml_raw['datasets']=[]
for target_resolution in target_resolutions:
  multipart_resolutions = target_resolution.split('||')
  multipart_batches = batch_size.split('||')
  if len(multipart_resolutions) > 1:
    if 'body' in image_path.lower():
      res = int(multipart_resolutions[0])
    else:
      res = int(multipart_resolutions[1])
  else:
    size_split = multipart_resolutions[0].split('x')
    
    res_width = int(size_split[0])
    if len(size_split) == 1:
      res_height = res_width
    else:
      res_height = int(size_split[1])

  if len(multipart_batches) > 1:
    if 'body' in image_path.lower():
      batch_num = int(multipart_batches[0])
    else:
      batch_num = int(multipart_batches[1])
  else:
    batch_num = int(multipart_batches[0])
  dataset={}
  dataset['enable_bucket'] = True
  dataset['resolution'] = [res_width,res_height]
  #dataset['min_bucket_reso'] = int(640 * min(res_width,res_height)/1024)
  #dataset['max_bucket_reso'] = int(1536 * max(res_width,res_height)/1024)
  dataset['min_bucket_reso'] = 512
  dataset['max_bucket_reso'] = 2048
  dataset['batch_size'] = batch_num
  dataset['bucket_reso_steps'] = 64
  dataset['subsets']=[]
  for image_path in image_part_path:
    subset={}
    subset['image_dir'] = image_path
    subset['num_repeats'] = repeat
    subset["shuffle_caption"] = shuffle
    subset["keep_tokens_separator"] = "|||"
    subset["keep_tokens"] = keep_token
    subset['class_tokens'] = class_token
    dataset['subsets'].append(subset)
  if use_reg and os.path.exists(reg_dataset_root):
    subset={}
    subset['image_dir'] = reg_dataset_root
    subset["keep_tokens"] = 0
    subset['num_repeats'] = reg_repeat
    subset["shuffle_caption"] = shuffle
    subset["keep_tokens_separator"] = "|||"
    subset['class_tokens'] = "landscape"
    dataset['subsets'].append(subset)
  toml_raw['datasets'].append(dataset)

with open(os.path.join(train_dir,"dataset.toml"),'w',encoding="utf-8") as f:
  toml.dump(toml_raw,f)

target_block_list = []
target_element_list = [x.strip() for x in Target_Elements.split(',')]
if Target_Blocks != "":
  import re
  def footage_element(blockname):
    return [f"{blockname}{x}" for x in target_element_list]

  target_block_arr = [x.strip() for x in Target_Blocks.split(',')]
  for target_block in target_block_arr:
    if 'mid' in target_block.lower():
      target_block_list.extend(footage_element("middle"))
    if 'out' in target_block.lower():
      block_number = re.search("[0-9]",target_block).group(0)
      target_block_list.extend(footage_element(f"output_blocks.{block_number}"))
    if 'in' in target_block.lower():
      block_number = re.search("[0-9]",target_block).group(0)
      target_block_list.extend(footage_element(f"input_blocks.{block_number}"))


lycoris_preset = Lycoris_preset
if Lycoris_preset=="custom":
  custom_dict = {}
  custom_dict["enable_conv"] = True
  custom_dict["text_encoder_target_module"] = [x.strip() for x in Text_Module.split(',')]
  custom_dict["text_encoder_target_name"] = [x.strip() for x in Text_Name.split(',')]
  custom_dict["unet_target_module"] = [x.strip() for x in Unet_Module.split(',')]
  if Unet_Name != "":
    custom_dict["unet_target_name"] = [x.strip() for x in Unet_Name.split(',')]
  elif len(target_block_list)>0:
    custom_dict["unet_target_name"] = target_block_list

  if Mapping_KV != "" and (Module_Maps != "" or Name_Maps != ""):
    values_dict={}
    for raw_kv in Mapping_KV.split():
      key, value = raw_kv.split("=")
      try:
          value = ast.literal_eval(value)
      except ValueError:
          pass
      values_dict[key] = value

    if Module_Maps != "":
      custom_dict["module_algo_map"] = {}
      for x in [x.strip() for x in Module_Maps.split(',')]:
        custom_dict["module_algo_map"][x] = {}
        custom_dict["module_algo_map"][x].update(values_dict)

    if Name_Maps != "":
      custom_dict["name_algo_map"] = {}
      for x in [x.strip() for x in Name_Maps.split(',')]:
        custom_dict["name_algo_map"][x] = {}
        custom_dict["name_algo_map"][x].update(values_dict)

  with open(os.path.join(train_dir,"lycoris_preset.toml"),'w',encoding="utf-8") as f:
    toml.dump(custom_dict,f)
  lycoris_preset = f"{train_dir}/lycoris_preset.toml"

In [ ]:
#@title  #RUN
!chcp 65001
import toml,os

SD_TYPE = "SDXL" #@param ["SD1.5","SDXL"]
TYPE = "LORA" #@param ["FINE_TUNE","LORA"]

Project_Name = "ryujin-62-LOHA32-RIJ18ILL-rj5_jf_wdcaption-shinryujin_girl-PDP_b9_995_ortho_eps1e16_nofac-TE_ATNT-LR1-b2g1r10-MR1_1024-CONST-modalkh-15E" #@param{type: 'string'}
#@markdown --------------------------------------------------
Gradient_Accum=1 #@param{type:'number'}
Max_Step = 0 #@param{type:'number'}
#@markdown * It overrides max step
Max_Epochs = 15 #@param {type:'number'}
Full_FP16 = False #@param{type:'boolean'}
#@markdown --------------------------------------------------
Main_Schedule = "constant" #@param ["constant","constant_with_warmup", "cosine", "cosine_with_min_lr", "cosine_with_restarts", "linear","polynomial","warmup_stable_decay"] {allow-input: true}
#@markdown * warmup / decay / cycles / min_lr_ratio / power / timescale
Main_LR_Param = "" #@param{type:'string'}
#@markdown * param 없을시 none입력. 빈칸일시 main따라감
TE1_Schedule = "none" #@param ["none","constant","constant_with_warmup", "cosine", "cosine_with_min_lr", "cosine_with_restarts", "linear","polynomial","warmup_stable_decay"] {allow-input: true}
TE1_LR_Param = "warmup=0.1" #@param{type:'string'}
TE2_Schedule = "none" #@param ["none","constant","constant_with_warmup", "cosine", "cosine_with_min_lr", "cosine_with_restarts", "linear","polynomial","warmup_stable_decay"] {allow-input: true}
TE2_LR_Param = "warmup=0.1" #@param{type:'string'}
#@markdown * it overrides lr_scheduler above
Custom_LR_Schedule = "" #@param ["","CosineAnnealingLR","OneCycleLR","library.lr_schedulers.REX"]{allow-input: true}

#@markdown * if you use mechanic, LR should be set to a high value. like 1
Use_Mechanic = False #@param{type:'boolean'}
Learning_Rate_TE = 1
Learning_Rate_TE2 = 1
Learning_Rate_Unet = 1
Optimizer = "prodigyplus.prodigy_plus_schedulefree.ProdigyPlusScheduleFree" #@param ["AdamW","AdamW8bit","Adafactor","AdamWScheduleFree","DAdaptAdam","Prodigy","ProdigyPlusScheduleFree","came_pytorch.CAME","pytorch_optimizer.StableAdamW"] {allow-input: true}
#@markdown * the format must be like betas=0.9,0.999 weight_decay=0.1 weight_lr_power=10
Optimizer_Args = "betas=0.9,0.995 weight_decay=0 eps=1e-16 use_orthograd=True factored=False" #@param {type:'string'}
#@markdown --------------------------------------------------
Debiased_Estimation = False #@param{type:"boolean"}
Masked_Loss = False #@param{type:"boolean"}
Masked_Latent = False #@param{type:"boolean"}
Masked_Random_Background = False #@param{type:"boolean"}
#@markdown *  --masked_loss_face_weight --masked_loss_body_weight --masked_loss_nonmask_weight --masked_loss_normalize --minimum_masked_loss_weight
Addional_Args = "--no_half_vae"  #@param {type:"string"}
Scale_Weight_Norms = 0 #@param{type:"number"}
MIN_SNR = 0 #@param{type:'number'}
Max_Grad_Norm = 0
Clip_Skip = 2 #@param{type:'number'}
#@markdown --------------------------------------------------
Min_Timestep = 0 #@param{type:'number'}
Max_Timestep = 1000 #@param{type:'number'}
Train_Noise_Scheduler="ddpm" #@param["ddpm","ddim","pndm","lms","edmeuler","flow_euler","euler","euler_a","heun","dpm_2","dpm_2_a","dpmsolver","dpmsolver++","sde-dpmsolver++","dpmsingle","dpmsde","deis"]
Train_Scheduler_Args = "" #@param {type:'string'}
Timestep_Sampling="uniform" #@param["uniform", "sigmoid", 'shift', "increase","decrease", "normal"]
Sigmoid_Bias = 0.0 #@param {type:'number'}
Sigmoid_Scale = 1.0 #@param {type:'number'}
Timestep_Shift = 1.0 #@param{type:'number'}
Loss_Type="l2" #@param["l2", "huber", "smooth_l1"]
Weighting_Scheme="sigma_sqrt" #@param["sigma_sqrt", "cosmap", "one", "none"]
#@markdown --------------------------------------------------
Sample_Unit = "NEVER" #@param ['EPOCH','STEP','NEVER']
Sample_Every = 150 #@param {type:'number'}
#@markdown
Save_Unit = "EPOCH" #@param ['EPOCH','STEP','NEVER']
Save_Every = 1 #@param {type:'number'}
Save_State = True #@param{type:'boolean'}
#@markdown --------------------------------------------------
network_module = "lycoris.kohya" #@param ['networks.lora','lycoris.kohya']
network_dim = 32 #@param {type:'number'}
network_alpha = 32 #@param {type:'number'}
Lora_Type = "LOHA" #@param ['LORA','LOHA','LOKR',"FULL","IA3","DYLORA","GLORA","DIAGOFT","BOFT"]
#@markdown * conv_dim=INT | conv_alpha=FLOAT | dora_wd=True
#@markdown * factor=8 | bypass_mode=False | train_norm=True  LOKR시 bypass_mode = False
Network_Args ="" #@param{type:'string'}
resume_from = r""

model_root=os.path.join(train_dir,"output/")

network_arguments = []
network_arguments.append(f"algo={Lora_Type.lower()}")
network_arguments.append(f"preset={lycoris_preset}")
for argument in  Network_Args.split():
  network_arguments.append(argument)

lora_train = f"{TYPE}" == "LORA"

if SD_TYPE == "SD1.5":
  EXECUTE = "train_network.py" if lora_train else "train_db.py"
else:
  EXECUTE = "sdxl_train_network.py" if lora_train else "sdxl_train.py"

print(f"TRAIN {EXECUTE}")

LORA_OPTIONS =""
if lora_train:
  LORA_OPTIONS = f"--clip_skip {Clip_Skip} --prior_loss_weight 1.0 --network_module {network_module} --network_dim {network_dim} --network_alpha {network_alpha}  --network_args {' '.join(network_arguments)} "
  if Learning_Rate_Unet > 0:
    LRS = f"--unet_lr {Learning_Rate_Unet} "
  else:
    LRS = "--network_train_text_encoder_only "
  if Learning_Rate_TE > 0 or Learning_Rate_TE2 > 0:
    LRS += f"--text_encoder_lr {Learning_Rate_TE} {Learning_Rate_TE2} "
  else:
    LRS += "--network_train_unet_only "
else:
  LRS = f"--learning_rate {Learning_Rate_Unet} "
  if Learning_Rate_TE > 0 or Learning_Rate_TE2 > 0:
    LRS += f"--learning_rate_te1 {Learning_Rate_TE} --learning_rate_te2 {Learning_Rate_TE2} --train_text_encoder "

ADDITIONAL_OPTIONS = ""

if TE1_Schedule != "none":
  ADDITIONAL_OPTIONS += f"--lr_scheduler_te1 {TE1_Schedule} "
  if TE1_LR_Param != "":
    ADDITIONAL_OPTIONS += f"--lr_scheduler_args_te1 {TE1_LR_Param} "

if TE2_Schedule != "none":
  ADDITIONAL_OPTIONS += f"--lr_scheduler_te2 {TE2_Schedule} "
  if TE2_LR_Param != "":
    ADDITIONAL_OPTIONS += f"--lr_scheduler_args_te2 {TE2_LR_Param} "

if Full_FP16:
  ADDITIONAL_OPTIONS += "--full_fp16 "
if Custom_LR_Schedule != "":
  ADDITIONAL_OPTIONS += f"--lr_scheduler_type {Custom_LR_Schedule} "
if Max_Epochs > 0 :
  ADDITIONAL_OPTIONS +=  f"--max_train_epochs {Max_Epochs} "
if MIN_SNR > 0:
  ADDITIONAL_OPTIONS +=  f"--min_snr_gamma {MIN_SNR} "
if Masked_Loss:
  ADDITIONAL_OPTIONS += "--masked_loss "
if Masked_Latent:
  ADDITIONAL_OPTIONS += "--masked_latents "
if Masked_Random_Background:
  ADDITIONAL_OPTIONS += "--masked_random_background "
if Debiased_Estimation:
   ADDITIONAL_OPTIONS += "--debiased_estimation_loss "
if Use_Mechanic:
  ADDITIONAL_OPTIONS += "--use_mechanic "
if Sample_Unit != "NEVER":
  if Sample_Unit == "EPOCH":
    ADDITIONAL_OPTIONS += f"--sample_every_n_epochs {Sample_Every} "
  elif Sample_Unit == "STEP":
    ADDITIONAL_OPTIONS += f"--sample_every_n_steps {Sample_Every} "
if Save_Unit != "NEVER":
  if Save_Unit == "EPOCH":
    ADDITIONAL_OPTIONS += f"--save_every_n_epochs {Save_Every} "
  elif Save_Unit == "STEP":
    ADDITIONAL_OPTIONS += f"--save_every_n_steps {Save_Every} "
if Save_State:
  ADDITIONAL_OPTIONS += "--save_state "
  ADDITIONAL_OPTIONS += "--save_last_n_epochs_state=1 "if Save_Unit=="EPOCH" else "--save_last_n_steps_state=1 "
if Addional_Args != "":
	ADDITIONAL_OPTIONS += f"{Addional_Args} "

if Main_LR_Param != "":
  lr_params = Main_LR_Param
  for kv in lr_params.split(" "):
    key, value = kv.split("=")
    set_up=False
    if key=="warmup":
      ADDITIONAL_OPTIONS += f"--lr_warmup_steps={value} "
      set_up=True
    elif key=="decay":
      ADDITIONAL_OPTIONS += f"--lr_decay_steps={value} "
      set_up=True
    elif key=="cycles":
      ADDITIONAL_OPTIONS += f"--lr_scheduler_num_cycles={value} "
      set_up=True
    elif key=="min_lr_ratio":
      ADDITIONAL_OPTIONS += f"--lr_scheduler_min_lr_ratio={value} "
      set_up=True
    elif key=="power":
      ADDITIONAL_OPTIONS += f"--lr_scheduler_power={value} "
      set_up=True
    if set_up:
      Main_LR_Param = Main_LR_Param.replace(kv,"").strip()

backup_project= f"{resume_from}"
if backup_project == "":
  backup_project = Project_Name
backup_directories=[]
backup_root_dir = backup_project
if os.path.exists(backup_root_dir):
  backup_directories = sorted(
                  [dirpath for dirpath in os.listdir(backup_root_dir) if
                  'state' in dirpath],
                  reverse=True,
                  )

if len(backup_directories) > 0:
  ADDITIONAL_OPTIONS += f"--resume {os.path.join(backup_root_dir,backup_directories[0])} --skip_until_initial_step "
  print(f"resume training from {os.path.join(backup_root_dir,backup_directories[0])}")

if Masked_Loss or Masked_Latent or Masked_Random_Background:
  data_toml = toml.load(os.path.join(root_dir,"dataset.toml"))
  for dataset in data_toml['datasets']:
    for subset in dataset['subsets']:
      subset['alpha_mask']=True
  with open(os.path.join(root_dir,"dataset.toml"),'w',encoding="utf-8") as f:
    toml.dump(data_toml,f)
else:
  data_toml = toml.load(os.path.join(root_dir,"dataset.toml"))
  for dataset in data_toml['datasets']:
    for subset in dataset['subsets']:
      if subset.get('conditioning_data_dir'):
        subset.pop('conditioning_data_dir')
      if subset.get('alpha_mask'):
        subset.pop('alpha_mask')
  with open(os.path.join(root_dir,"dataset.toml"),'w',encoding="utf-8") as f:
    toml.dump(data_toml,f)

#엑셀레이터 말고 파이썬으로 둘것!(왠진 모르겠는데 개느림)
os.chdir(root_dir)
dd = f'python {EXECUTE} \
  {LORA_OPTIONS} \
  {LRS} \
  {ADDITIONAL_OPTIONS} \
  --seed 9123 \
  --cache_latents \
  --lowram \
  --caption_extension ".txt" \
  --caption_separator "," \
  --max_grad_norm {Max_Grad_Norm} \
  --max_train_steps {Max_Step} \
  --gradient_accumulation_steps {Gradient_Accum} \
  --gradient_checkpointing \
  --persistent_data_loader_workers \
  --huber_c 0.1 \
  --huber_schedule "snr" \
  --scale_weight_norms {Scale_Weight_Norms} \
  --loss_type {Loss_Type} \
  --lr_scheduler {Main_Schedule} \
  --lr_scheduler_args {Main_LR_Param} \
  --min_timestep {Min_Timestep} \
  --max_timestep {Max_Timestep} \
  --mixed_precision "bf16" \
  --optimizer_type {Optimizer} \
  --optimizer_args {Optimizer_Args} \
  --output_dir "{model_root}" \
  --logging_dir "{model_root}{Project_Name}/logs" \
  --output_name {Project_Name} \
  --pretrained_model_name_or_path {nai} \
  --sample_sampler "euler" \
  --save_model_as "safetensors" \
  --save_precision "bf16" \
  --dataset_config "{train_dir}/dataset.toml" \
  --sdpa \
  --vae_batch_size 1'
print("----------local----------")
print(dd)
modal_text = f'{EXECUTE} \
  {LORA_OPTIONS} \
  {LRS} \
  {ADDITIONAL_OPTIONS} \
  --seed 9123 \
  --cache_latents \
  --lowram \
  --caption_extension .txt \
  --caption_separator , \
  --max_grad_norm {Max_Grad_Norm} \
  --max_train_steps {Max_Step} \
  --gradient_accumulation_steps {Gradient_Accum} \
  --gradient_checkpointing \
  --persistent_data_loader_workers \
  --huber_c 0.1 \
  --huber_schedule snr \
  --scale_weight_norms {Scale_Weight_Norms} \
  --loss_type {Loss_Type} \
  --lr_scheduler {Main_Schedule} \
  --lr_scheduler_args {Main_LR_Param} \
  --min_timestep {Min_Timestep} \
  --max_timestep {Max_Timestep} \
  --mixed_precision bf16 \
  --optimizer_type {Optimizer} \
  --optimizer_args {Optimizer_Args} \
  --output_dir "{model_root}" \
  --logging_dir {model_root}{Project_Name}/logs \
  --output_name {Project_Name} \
  --pretrained_model_name_or_path {nai} \
  --sample_sampler euler \
  --save_model_as safetensors \
  --save_precision bf16 \
  --dataset_config {train_dir}/dataset.toml \
  --sdpa \
  --console_log_file /root/output/log.txt \
  --vae_batch_size 1'
print("----------modal----------")
with open("modal_command.txt","w",encoding="utf-8") as f:
  f.write(modal_text)
print(f"modal run run_modal.py --commendtxt=modal_command.txt")

Active code page: 65001
TRAIN sdxl_train_network.py
----------local----------
python sdxl_train_network.py   --clip_skip 2 --prior_loss_weight 1.0 --network_module lycoris.kohya --network_dim 32 --network_alpha 32  --network_args algo=loha preset=attn-mlp    --unet_lr 1 --text_encoder_lr 1 1    --max_train_epochs 5 --save_every_n_epochs 1 --save_state --save_last_n_epochs_state=1 --no_half_vae    --seed 9123   --cache_latents   --lowram   --caption_extension ".txt"   --caption_separator ","   --max_grad_norm 0   --max_train_steps 0   --gradient_accumulation_steps 1   --gradient_checkpointing   --persistent_data_loader_workers   --huber_c 0.1   --huber_schedule "snr"   --scale_weight_norms 0   --loss_type l2   --lr_scheduler constant   --lr_scheduler_args    --min_timestep 0   --max_timestep 1000   --mixed_precision "bf16"   --optimizer_type prodigyplus.prodigy_plus_schedulefree.ProdigyPlusScheduleFree   --optimizer_args betas=0.9,0.995 weight_decay=0 eps=1e-16 use_orthograd=True factor